# 7B block-level paper: GPU experiments E2–E4 (free Colab T4), v2

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked.
3. Within about a minute, cell 3 prints **"Preflight OK"**. Then it loads each model and runs the experiments (about 2 hours).
4. When it prints **ALL MODELS DONE**, cell 5 downloads `results.jsonl`. Upload that file to the chat.

**Please do not edit cells or accept Colab's AI "fix" suggestions.** If an error appears, copy it into the chat.

Results are saved to `MyDrive/arxiv_7b_experiments/results.jsonl` as they finish. If Colab disconnects, reconnect and **Run all** again. Finished items are skipped.

In [1]:
# 1) Mount Google Drive (results are saved there)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%%writefile /content/t4_experiments.py
#!/usr/bin/env python3
"""GPU experiments E2-E4 for the 7B block-level paper (free Colab T4).

Run in Colab:   !python /content/t4_experiments.py
Results go to:  /content/drive/MyDrive/arxiv_7b_experiments/results.jsonl  (one line per result, resumable)

Do not edit this file. If anything fails, copy the error and send it back.
"""
import importlib
import os
import subprocess
import sys

sys.stdout.reconfigure(line_buffering=True)  # live output under `!python` in Colab

PINS = {"transformers": "5.22.2", "accelerate": "1.15.0"}


def ensure_packages():
    """Install the exact library versions this code was tested with, then restart the process once."""
    if os.environ.get("T4EXP_SKIP_ENSURE"):
        return
    need = []
    for pkg, ver in PINS.items():
        try:
            if importlib.import_module(pkg).__version__ != ver:
                need.append(f"{pkg}=={ver}")
        except ImportError:
            need.append(f"{pkg}=={ver}")
    try:
        importlib.import_module("bitsandbytes")
    except ImportError:
        need.append("bitsandbytes>=0.48")
    if need:
        if os.environ.get("T4EXP_REEXEC"):
            sys.exit(f"Package install did not take effect: {need}")
        print("Installing pinned packages:", need, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
        os.environ["T4EXP_REEXEC"] = "1"
        os.execv(sys.executable, [sys.executable] + sys.argv)


ensure_packages()

import gc
import json
import random
import statistics
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DynamicCache

CHUNK = 1024          # prefill chunk size (must stay below Mistral's 4,096 window)
DECODE_STEPS = 32     # decode steps timed per E2 context length
DEPTHS = [0.1, 0.3, 0.5, 0.7, 0.9]
TRIALS_PER_DEPTH = 2  # E3 trials per (length, depth); 2 -> 10 trials per length

# Standard passkey-retrieval prompt (Mohtashami & Jaggi, 2023, "Landmark Attention").
INTRO = ("There is an important info hidden inside a lot of irrelevant text. "
         "Find it and memorize it. I will quiz you about the important information there.\n\n")
FILLER = "The grass is green. The sky is blue. The sun is yellow. Here we go. There and back again. "
NEEDLE = "The pass key is {key}. Remember it. {key} is the pass key. "
QUESTION = "\n\nWhat is the pass key? The pass key is"


def _dev(model):
    return next(model.parameters()).device


def _sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()


def _mem(dev, peak=False):
    if dev.type != "cuda":
        return 0
    return torch.cuda.max_memory_allocated() if peak else torch.cuda.memory_allocated()


def _free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model(repo, quantize=True):
    tok = AutoTokenizer.from_pretrained(repo)
    kw = dict(dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if quantize:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(repo, **kw).eval()
    return tok, model


def cache_stats(cache):
    """Bytes actually held by the KV cache tensors, and the max number of token positions held."""
    total, held = 0, 0
    for layer in cache.layers:
        for t in (layer.keys, layer.values):
            if isinstance(t, torch.Tensor):
                total += t.numel() * t.element_size()
        if isinstance(layer.keys, torch.Tensor):
            held = max(held, layer.keys.shape[-2])
    return total, held


@torch.no_grad()
def prefill(model, ids, chunk=CHUNK):
    cache, logits = DynamicCache(config=model.config), None
    for s in range(0, ids.shape[1], chunk):
        out = model(input_ids=ids[:, s:s + chunk], past_key_values=cache, use_cache=True, logits_to_keep=1)
        cache, logits = out.past_key_values, out.logits[:, -1].float()
    return cache, logits


@torch.no_grad()
def greedy(model, cache, logits, steps, stop_ids=()):
    dev, toks, times = _dev(model), [], []
    nxt = logits.argmax(-1, keepdim=True)
    for i in range(steps):
        toks.append(int(nxt.item()))
        if toks[-1] in stop_ids or i == steps - 1:
            break
        _sync(dev)
        t0 = time.perf_counter()
        out = model(input_ids=nxt, past_key_values=cache, use_cache=True, logits_to_keep=1)
        _sync(dev)
        times.append(time.perf_counter() - t0)
        cache = out.past_key_values
        nxt = out.logits[:, -1].argmax(-1, keepdim=True)
    return toks, times


def load_done(path):
    done = set()
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            if line.strip():
                done.add(json.loads(line)["key"])
    return done


def append(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")
    print({k: v for k, v in row.items() if k not in ("output",)})


def sanity_check(name, tok, model, out_path, done):
    key = f"sanity|{name}"
    if key in done:
        return
    dev = _dev(model)
    ids = tok("The capital of France is", return_tensors="pt").input_ids.to(dev)
    cache, logits = prefill(model, ids)
    toks, _ = greedy(model, cache, logits, 8)
    append(out_path, dict(key=key, exp="sanity", model=name, finite=bool(torch.isfinite(logits).all()),
                          output=tok.decode(toks)))


def run_e2(name, tok, model, contexts, out_path, done, seed=0):
    """E2/E4: KV-cache bytes, device memory, prefill time and decode latency versus context length."""
    dev = _dev(model)
    g = torch.Generator().manual_seed(seed)
    lo, hi = 300, tok.vocab_size  # regular (non-special) token ids
    _free()
    w_cache, w_logits = prefill(model, torch.randint(lo, hi, (1, 256), generator=g).to(dev))  # warm-up
    greedy(model, w_cache, w_logits, 4)
    del w_cache, w_logits
    _free()
    base = _mem(dev)
    for T in contexts:
        key = f"e2|{name}|{T}"
        if key in done:
            continue
        _free()
        if dev.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        ids = torch.randint(lo, hi, (1, T), generator=g).to(dev)
        row, oom = dict(key=key, exp="e2", model=name, ctx=T), False
        try:
            _sync(dev)
            t0 = time.perf_counter()
            cache, logits = prefill(model, ids)
            _sync(dev)
            prefill_s = time.perf_counter() - t0
            kv_bytes, held = cache_stats(cache)
            after = _mem(dev) - base
            toks, times = greedy(model, cache, logits, DECODE_STEPS)
            row.update(status="ok", kv_cache_mib=kv_bytes / 2**20, kv_positions_held=held,
                       alloc_after_prefill_mib=after / 2**20, peak_over_weights_mib=(_mem(dev, True) - base) / 2**20,
                       prefill_s=prefill_s, prefill_tok_per_s=T / prefill_s,
                       decode_ms_per_token=1000 * statistics.median(times) if times else None,
                       logits_finite=bool(torch.isfinite(logits).all()))
            del cache, logits
        except torch.OutOfMemoryError:
            oom = True
        if oom:
            _free()
            row.update(status="oom")
        del ids
        _free()
        append(out_path, row)
        if oom:
            break  # longer contexts will not fit either


def passkey_prompt(tok, T, depth, key):
    """Build a passkey prompt of at most T tokens (this model's tokenizer) with the needle at `depth`."""
    def n_tokens(s):
        return len(tok(s).input_ids)
    per = (n_tokens(FILLER * 40) - n_tokens(FILLER * 20)) / 20
    fixed = n_tokens(INTRO + NEEDLE.format(key=key) + QUESTION)
    n = max(1, int((T - fixed) / per))
    while True:
        k = int(round(depth * n))
        text = INTRO + FILLER * k + NEEDLE.format(key=key) + FILLER * (n - k) + QUESTION
        ids = tok(text, return_tensors="pt").input_ids
        if ids.shape[1] <= T or n <= 1:
            return ids, k / n
        n -= max(1, int((ids.shape[1] - T) / per))


def run_e3(name, tok, model, contexts, out_path, done):
    """E3: passkey retrieval accuracy versus context length and needle depth."""
    dev = _dev(model)
    stop = {i for i in (tok.eos_token_id,) if i is not None}
    for T in contexts:
        for depth in DEPTHS:
            for trial in range(TRIALS_PER_DEPTH):
                key = f"e3|{name}|{T}|{depth}|{trial}"
                if key in done:
                    continue
                pk = random.Random(f"{T}-{depth}-{trial}").randint(10000, 99999)  # same keys for every model
                ids, real_depth = passkey_prompt(tok, T, depth, pk)
                row, oom = dict(key=key, exp="e3", model=name, ctx=T, ctx_actual=int(ids.shape[1]),
                                depth=depth, depth_actual=real_depth, trial=trial, passkey=pk), False
                try:
                    t0 = time.perf_counter()
                    cache, logits = prefill(model, ids.to(dev))
                    toks, _ = greedy(model, cache, logits, 12, stop)
                    text = tok.decode(toks)
                    digits = "".join(ch if ch.isdigit() else " " for ch in text).split()
                    row.update(status="ok", output=text, correct=bool(digits) and digits[0] == str(pk),
                               seconds=time.perf_counter() - t0)
                    del cache, logits
                except torch.OutOfMemoryError:
                    oom = True
                if oom:
                    _free()
                    row.update(status="oom", correct=False)
                _free()
                append(out_path, row)


# ---------------------------------------------------------------- run plan
import platform  # noqa: E402
import shutil  # noqa: E402
import tempfile  # noqa: E402

MODELS = {
    "mistral-7b": dict(repo="mistralai/Mistral-7B-v0.1", e2=[1024, 2048, 4096, 8192, 16384, 32768, 65536],
                       e3=[4096, 8192, 16384, 32768]),
    "qwen2.5-7b": dict(repo="Qwen/Qwen2.5-7B", e2=[1024, 2048, 4096, 8192, 16384, 32768, 65536],
                       e3=[4096, 8192, 16384, 32768]),
    "llama-7b": dict(repo="huggyllama/llama-7b", e2=[1024, 2048, 4096, 8192, 16384, 32768],
                     e3=[1024, 2048, 4096]),
}
OUT = Path(os.environ.get("T4EXP_OUT", "/content/drive/MyDrive/arxiv_7b_experiments/results.jsonl"))


def read_rows():
    return [json.loads(l) for l in OUT.read_text().splitlines() if l.strip()] if OUT.exists() else []


def plan(name, cfg, rows):
    keys = {r["key"] for r in rows}
    ooms = [r["ctx"] for r in rows if r.get("exp") == "e2" and r.get("model") == name and r.get("status") == "oom"]
    e2 = [T for T in cfg["e2"] if (not ooms or T < min(ooms)) and f"e2|{name}|{T}" not in keys]
    e3 = [T for T in cfg["e3"]
          if any(f"e3|{name}|{T}|{d}|{t}" not in keys for d in DEPTHS for t in range(TRIALS_PER_DEPTH))]
    return (f"sanity|{name}" not in keys), e2, e3


def preflight():
    """Seconds-long check that 4-bit loading and the prefill/decode loop work on this GPU, before any download."""
    from transformers import MistralConfig, MistralForCausalLM
    d = tempfile.mkdtemp()
    MistralForCausalLM(MistralConfig(vocab_size=1000, hidden_size=256, intermediate_size=512, num_hidden_layers=2,
                                     num_attention_heads=4, num_key_value_heads=1, sliding_window=64)).save_pretrained(d)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForCausalLM.from_pretrained(d, dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",
                                             quantization_config=q).eval()
    assert any(type(x).__name__ == "Linear4bit" for x in m.modules()), "4-bit quantization was not applied"
    cache, logits = prefill(m, torch.randint(0, 1000, (1, 300), device="cuda"), chunk=128)
    greedy(m, cache, logits, 4)
    assert cache_stats(cache)[1] == 63, "sliding-window cache check failed"
    del m, cache, logits
    _free()
    shutil.rmtree(d, ignore_errors=True)
    print("Preflight OK: 4-bit loading, chunked prefill, decode and sliding-window cache all work on this GPU.", flush=True)


def check_quantized(model):
    n4 = sum(type(x).__name__ == "Linear4bit" for x in model.modules())
    gib = torch.cuda.memory_allocated() / 2**30
    if n4 == 0 or gib > 8:
        raise SystemExit(f"Model is not 4-bit quantized (Linear4bit layers={n4}, weights={gib:.1f} GiB). Stopping.")
    return n4, gib


def gpu_ok():
    return torch.cuda.is_available()


def gpu_info():
    return dict(gpu=torch.cuda.get_device_name(0),
                gpu_mem_gib=round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


def main():
    if not gpu_ok():
        sys.exit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then run again.")
    if not OUT.parent.parent.exists():
        sys.exit("Google Drive is not mounted. Run: from google.colab import drive; drive.mount('/content/drive')")
    OUT.parent.mkdir(parents=True, exist_ok=True)
    import accelerate
    import bitsandbytes
    import transformers
    meta = dict(key=f"meta|{time.strftime('%Y-%m-%dT%H:%M:%S')}", exp="meta", **gpu_info(),
                torch=torch.__version__, cuda=torch.version.cuda, transformers=transformers.__version__,
                bitsandbytes=bitsandbytes.__version__, accelerate=accelerate.__version__,
                python=platform.python_version())
    append(OUT, meta)
    preflight()
    for name, cfg in MODELS.items():
        need_sanity, e2_todo, e3_todo = plan(name, cfg, read_rows())
        if not (need_sanity or e2_todo or e3_todo):
            print(f"== {name}: already complete, skipping", flush=True)
            continue
        t0 = time.time()
        print(f"== {name}: loading {cfg['repo']} (uses the download cache if present)", flush=True)
        tok, model = load_model(cfg["repo"])
        n4, gib = check_quantized(model)
        print(f"   loaded in {time.time() - t0:.0f}s; {n4} Linear4bit layers; weights use {gib:.2f} GiB", flush=True)
        done = load_done(OUT)
        sanity_check(name, tok, model, OUT, done)
        run_e2(name, tok, model, e2_todo, OUT, done)
        run_e3(name, tok, model, e3_todo, OUT, done)
        del model
        _free()
        from huggingface_hub import constants
        shutil.rmtree(Path(constants.HF_HUB_CACHE) / ("models--" + cfg["repo"].replace("/", "--")), ignore_errors=True)
        print(f"== {name}: done in {(time.time() - t0) / 60:.1f} min", flush=True)
    rows = read_rows()
    print("\nALL MODELS DONE.", sum(r.get("exp") in ("e2", "e3", "sanity") for r in rows), "result rows in", OUT)


if __name__ == "__main__":
    main()


Writing /content/t4_experiments.py


In [3]:
# 3) Run: preflight check on the GPU first, then Mistral, Qwen2.5, LLaMA
!python /content/t4_experiments.py


Installing pinned packages: ['transformers==5.22.2', 'bitsandbytes>=0.48']
{'key': 'meta|2026-11-13T01:34:00', 'exp': 'meta', 'gpu': 'Tesla T4', 'gpu_mem_gib': 14.56, 'torch': '2.14.1+cu130', 'cuda': '13.0', 'transformers': '5.22.2', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0', 'python': '3.13.16'}
Writing model shards: 100% 1/1 [00:00<00:00, 164.94it/s]
Loading weights: 100% 21/21 [00:00<00:00, 242.82it/s]
Preflight OK: 4-bit loading, chunked prefill, decode and sliding-window cache all work on this GPU.
== mistral-7b: loading mistralai/Mistral-7B-v0.1 (uses the download cache if present)
config.json: 100% 571/571 [00:00<00:00, 3.33MB/s]
tokenizer_config.json: 100% 996/996 [00:00<00:00, 5.58MB/s]
tokenizer.json: 100% 1.80M/1.80M [00:00<00:00, 63.1MB/s]

tokenizer.model: downloading bytes:   8% 41.5k/493k [00:00<00:05, 83.2kB/s]
tokenizer.model: downloading bytes: 100% 339k/339k [00:00<00:00, 598kB/s, 33.3kB/s  ] 
tokenizer.model: reconstructing file: 100% 493k/493k [00:00<00:00, 

In [4]:
# 4) Quick summary
import json, pandas as pd
from pathlib import Path
OUT = Path('/content/drive/MyDrive/arxiv_7b_experiments/results.jsonl')
df = pd.DataFrame([json.loads(l) for l in OUT.read_text().splitlines() if l.strip()])
e2, e3 = df[df.exp == 'e2'], df[df.exp == 'e3']
print(e2.pivot_table(index='ctx', columns='model', values='kv_cache_mib').round(1), '\n')
print(e2.pivot_table(index='ctx', columns='model', values='decode_ms_per_token').round(1), '\n')
print(e2[e2.status == 'oom'][['model', 'ctx', 'status']], '\n')
print(e3.pivot_table(index='ctx', columns='model', values='correct', aggfunc='mean').round(2))


model    llama-7b  mistral-7b  qwen2.5-7b
ctx                                      
1024.0      512.0       128.0        56.0
2048.0     1024.0       256.0       112.0
4096.0     2048.0       511.9       224.0
8192.0     4096.0       511.9       448.0
16384.0    8192.0       511.9       896.0
32768.0       NaN       511.9      1792.0
65536.0       NaN       511.9      3584.0 

model    llama-7b  mistral-7b  qwen2.5-7b
ctx                                      
1024.0       47.4        48.0        49.6
2048.0       49.1        48.8        53.5
4096.0       68.3        83.4        57.4
8192.0       91.1        83.7        66.7
16384.0     148.5        82.2        80.2
32768.0       NaN        83.6       116.7
65536.0       NaN        83.0       187.4 

        model      ctx status
104  llama-7b  32768.0    oom 

model   llama-7b mistral-7b qwen2.5-7b
ctx                                   
1024.0       1.0        NaN        NaN
2048.0       1.0        NaN        NaN
4096.0       0.0      

In [5]:
# 5) Download the results file, then upload it to the chat
from google.colab import files
files.download('/content/drive/MyDrive/arxiv_7b_experiments/results.jsonl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>